# Step 5.2: Tweet tickers × trade tickers

The goal of this notebook is one dataset of Trump's social-media company references in 2025 and 2026 where every company he named is classified: its stock ticker if it is publicly listed (`public_ticker`), the same ticker as `ticker` if the company is in the trade universe (`company.db`, Step 4), and whether he purchased it (`purchased`, from `purchases.csv`). Part 1 of the thesis needs exactly this: the share of listed companies he posted about that he also bought. The LLM (Step 5.1) gave us a company name and a ticker guess per reference; here we decide the company's ticker the same way Step 2 decided tickers for the disclosures: deterministic rules give a suggestion, and a person reviews every company in `TickerTwitterReview.xlsx`. Nothing in this notebook reads the trade dates. The logic flows as follows:

1. Data import and standardizing names and tickers
2. One suggested ticker per company
3. Review of every company (`TickerTwitterReview.xlsx`) and applying the decisions
4. Final dataset
5. Descriptive analysis

In [1]:
import re
import os
import json
import sqlite3
import pandas as pd
import numpy as np
from rapidfuzz import fuzz, process
from openpyxl import load_workbook, Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation

SOCIAL_DB = "../1) LabelTweets/SocialMedia.db"
COMPANY_DB = "../../4) CompanyDB/company.db"
TRADES_REVIEW = "../../2) Trades/TickerReview.xlsx"
PRICES_CSV = "../../3) Prices/1) FetchPrices/prices.csv"   # daily prices of the universe (Step 3); used for the listing check in 3.4
PURCHASES_CSV = "../../3) Prices/RawData/purchases.csv"      # the final purchases (Step 2.6): the numerator of Part 1
SYMBOL_CHANGES = "../../2) Trades/RawData/FMP_symbol_changes.json"   # FMP's symbol-change list (Step 2.5 cache): symbols reassigned to another company
review_path = "TickerTwitterReview.xlsx"

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

## 1) Data import and standardizing names and tickers

We import every company reference from Step 5.1 together with its post, and the ticker universe from Step 4. Both sides then get the same name and ticker standardization so that a name or ticker means the same thing on both sides.

### 1.1) Data import

One row per post and company reference (`SocialMediaKeywords`), joined to the post it came from (`SocialMedia`) for the date, account and text. The universe is `companies` (one row per traded ticker with its two FMP names) plus `executives` (one row per ticker and executive), used only to find the company behind an executive reference.

In [2]:
with sqlite3.connect(SOCIAL_DB) as conn:
    refs = pd.read_sql_query(
        """SELECT k.rowid AS ref_id, k.ID, s.created_at, s.User, s.platform, s.text,
                  k.mention_text, k.entity_type, k.canonical_company, k.ticker_guess, k.reference_type,
                  k.sentiment, k.relevance, k.confidence, s.llm_model, s.llm_prompt_version
           FROM SocialMediaKeywords k JOIN SocialMedia s ON s.ID = k.ID
           ORDER BY s.created_at, k.ID, k.rowid""", conn)
    posts_total = pd.read_sql_query("SELECT llm_status, count(*) AS n FROM SocialMedia GROUP BY 1", conn)

with sqlite3.connect(COMPANY_DB) as conn:
    companies = pd.read_sql_query("SELECT ticker, fmp_company_name, profile_company_name, sector, industry FROM companies", conn)
    executives = pd.read_sql_query("SELECT ticker, name FROM executives", conn)

refs["post_date"] = refs["created_at"].str[:10]
print(f"References loaded: {len(refs)} rows from {refs['ID'].nunique()} posts")
print(f"Universe loaded: {len(companies)} tickers, {len(executives)} executive rows")
print("\nPosts by LLM status:")
display(posts_total)
print("\nReferences by reference_type:")
display(refs["reference_type"].value_counts().rename_axis("reference_type").reset_index(name="rows"))

References loaded: 6468 rows from 4376 posts
Universe loaded: 1620 tickers, 16777 executive rows

Posts by LLM status:


,llm_status,n
0,content_filter,331
1,ok,21942



References by reference_type:


,reference_type,rows
0,indirect,3649
1,direct,2246
2,brand,412
3,executive,161


### 1.2) Standardizing company names

The Step 2.2 standardization ("Corporation" → "CORP", "Incorporated" → "INC", "Public Limited Company" → "PLC", upper case, dots and commas stripped) is applied to the LLM's `canonical_company` and to the names of the universe: both FMP names and the SEC names the ticker's registrant has carried since 2020 (Step 4.2 entity sources), so that "IAC Inc." still meets `PPLI`, today People Inc. From the universe we build one lookup from standardized name to the set of tickers that carry it; a name that fits more than one ticker is never resolved by the rules (`ambiguous` in section 2).

The standardized LLM name is also the key of the review: two spellings that standardize to the same name ("AstraZeneca PLC", "AstraZeneca plc") are one company and get one decision.

In [3]:
def standardize_company(name) -> str:
    if name is None or (isinstance(name, float) and np.isnan(name)):
        return ""
    s = str(name).upper().strip()
    s = s.replace(".", "").replace(",", "")
    s = re.sub(r"\bINCORPORATED\b", "INC", s)
    s = re.sub(r"\bCORPORATION\b", "CORP", s)
    s = re.sub(r"\bPUBLIC LIMITED COMPANY\b", "PLC", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

refs["CompanyStd"] = refs["canonical_company"].apply(standardize_company)
companies["FmpStd"] = companies["fmp_company_name"].apply(standardize_company)
companies["ProfileStd"] = companies["profile_company_name"].apply(standardize_company)

# SEC current and former names of each universe ticker's registrant (Step 4.2 cache), standardized
with sqlite3.connect(COMPANY_DB) as conn:
    cik_of_ticker = dict(conn.execute("SELECT ticker, entity_id FROM entities WHERE in_universe = 1 AND entity_id IS NOT NULL"))
    sec_payloads = dict(conn.execute("SELECT key, payload FROM entity_sources WHERE source = 'sec_submissions'"))
former_names = {}
for t, cik in cik_of_ticker.items():
    sub = json.loads(sec_payloads.get(cik) or "null") or {}
    # former names held in or after 2020 only: an older name (TEGNA was "Gannett Co., Inc." until 2015) belongs to another company today
    names = {standardize_company(sub.get("name"))} | {standardize_company(f.get("name")) for f in sub.get("formerNames", [])
                                                      if (f.get("to") or "")[:10] >= "2020-01-01"}
    former_names[t] = names - {""}
print(f"SEC names loaded for {len(former_names)} tickers ({sum(len(v) for v in former_names.values())} current and former names held since 2020)")

# standardized name -> set of tickers (over both FMP names and the SEC names), and the reverse
name_to_tickers = {}
for _, r in companies.iterrows():
    for n in {r["FmpStd"], r["ProfileStd"]} | former_names.get(r["ticker"], set()):
        if n:
            name_to_tickers.setdefault(n, set()).add(r["ticker"])
ticker_to_names = {r["ticker"]: ({r["FmpStd"], r["ProfileStd"]} | former_names.get(r["ticker"], set())) - {""} for _, r in companies.iterrows()}
ticker_to_fmp = companies.set_index("ticker")["profile_company_name"].to_dict()

ambiguous_names = {n: t for n, t in name_to_tickers.items() if len(t) > 1}
print(f"{len(name_to_tickers)} distinct standardized universe names; {len(ambiguous_names)} fit more than one ticker:")
for n, t in sorted(ambiguous_names.items()):
    print(f"   {n}: {sorted(t)}")

SEC names loaded for 1618 tickers (1910 current and former names held since 2020)


2177 distinct standardized universe names; 2 fit more than one ticker:
   FIRST BANCORP: ['FBNC', 'FBP']
   IAC/INTERACTIVECORP: ['MTCH', 'PPLI']


### 1.3) Standardizing ticker guesses

The LLM writes the primary U.S. ticker as it knows it. The universe uses the FMP convention of Step 2.4 (one class per company, share classes with a hyphen), so the guess is upper-cased, stripped of `$`, dots and spaces become hyphens, and two maps are applied, listed in full here:

* **share-class map**: the class the model prefers → the class in the universe (`GOOGL` → `GOOG`, `NWS` → `NWSA`, `BRK-A` → `BRK-B`);
* **rename map**: the same chains Step 2.5 applies to the trades, so a guess under an old symbol meets the company's latest symbol. They come from `2) Trades/TickerReview.xlsx`, sheet `TickerRenames`, rows with `Decision = apply`, plus `PARA` → `PSKY` (Paramount Global became Paramount Skydance on 2025-08-07), followed to their end (`PARA` → `PSKY` → `SKYD`).

The rename map holds only true renames of the same company. A link between two different companies is not a rename and does not belong here (the earlier `EDR` → `TKO` entry did: Endeavor and TKO are different companies, `tasks/lessons.md` 2026-11-03); such a case is decided in the review, where it is visible with its reason. The original guess is kept in `ticker_guess`; the standardized one is `TickerGuessStd`.

A symbol is a label with a date, not a company: exchanges reassign retired symbols. `PARA` was Paramount Global until the Skydance merger on 2025-08-07 and has belonged to Parabolic Technologies since 2026-08-07, inside our sample. The price provider serves every series under the company's newest symbol, so the pipeline never looks an old symbol up (SKYD's series is Paramount's history), but a guess under a reassigned symbol dated after the reassignment would be sent through the chain to the wrong company. The notebook therefore reads FMP's symbol-change list and stops if any guess under an old symbol of the rename map is dated on or after the day that symbol was given to another company; those references would have to be decided in the review, not by the map. Today there are none: the 81 `PARA` guesses run to 2026-07-17.

In [4]:
SHARE_CLASS_MAP = {"GOOGL": "GOOG", "NWS": "NWSA", "BRK-A": "BRK-B", "BRKA": "BRK-B", "BF-A": "BF-B"}
c8 = pd.read_excel(TRADES_REVIEW, sheet_name="TickerRenames")
RENAME_MAP = dict(zip(c8.loc[c8["Decision"] == "apply", "OldTicker"], c8.loc[c8["Decision"] == "apply", "LatestTicker"]))
RENAME_MAP.update({"PARA": "PSKY"})

def follow_renames(s):
    seen = set()
    while s in RENAME_MAP and s not in seen:
        seen.add(s); s = RENAME_MAP[s]
    return s

def standardize_ticker(t):
    if t is None or (isinstance(t, float) and np.isnan(t)) or str(t).strip() == "":
        return None
    s = str(t).upper().strip().lstrip("$")
    s = re.sub(r"[.\s/]+", "-", s)
    s = SHARE_CLASS_MAP.get(s, s)
    return follow_renames(s)

print(f"rename map: {len(RENAME_MAP)} entries:", ", ".join(f"{k}->{follow_renames(k)}" for k in sorted(RENAME_MAP)))

refs["TickerGuessStd"] = refs["ticker_guess"].apply(standardize_ticker)
universe = set(companies["ticker"])

# symbols of the rename map that FMP later gave to another company: a guess under such a symbol, dated after the reassignment, must not follow the chain
symbol_changes = pd.DataFrame(json.load(open(SYMBOL_CHANGES, encoding="utf-8"))["rows"])   # date, companyName, oldSymbol, newSymbol
raw_guess = refs["ticker_guess"].str.upper().str.strip().str.lstrip("$")
retired_on = {**{o: str(d)[:10] for o, d in zip(c8["OldTicker"], c8["ChangeDate"]) if o in RENAME_MAP and str(d)[:4].isdigit()}, "PARA": "2025-08-07"}
reassigned = {}
for old in RENAME_MAP:
    later = symbol_changes[(symbol_changes["newSymbol"] == old) & (symbol_changes["date"] > retired_on.get(old, "9999"))]   # given to another company after our chain left it
    if len(later):
        reassigned[old] = (later["date"].min(), later.sort_values("date")["companyName"].iat[0])
print("Rename-map symbols later reassigned to another company:", {k: f"{v[1]} from {v[0]}" for k, v in reassigned.items()})
after_reuse = refs[raw_guess.isin(reassigned) & (refs["post_date"] >= raw_guess.map(lambda g: reassigned.get(g, ("9999",))[0]))]
assert after_reuse.empty, f"guesses under a reassigned symbol, dated after the reassignment; decide them in the review: {after_reuse[['ID', 'post_date', 'ticker_guess', 'canonical_company']].values.tolist()}"
print(f"Guesses under a reassigned symbol dated after the reassignment: {len(after_reuse)}")

changed = refs[refs["ticker_guess"].notna() & (refs["ticker_guess"].str.upper() != refs["TickerGuessStd"])]
print(f"Ticker guesses: {refs['ticker_guess'].notna().sum()} rows with a guess, {refs['TickerGuessStd'].isin(universe).sum()} of them in the universe after standardization")
print(f"Guesses changed by the standardization or the maps: {len(changed)} rows")
display(changed.groupby(["ticker_guess", "TickerGuessStd"]).size().reset_index(name="rows").sort_values("rows", ascending=False))

rename map: 22 entries: AMBC->OSG, ASGN->EFOR, ATGE->CVSA, AXL->DCH, BHLB->BBT, BK->BNY, CSWI->CSW, EQR->VMRK, HONAV->HONA, IAC->PPLI, MGPI->CASK, MMC->MRSH, MODG->CALY, MSGS->MSGK, NYMT->ADAM, OMI->ACH, PARA->SKYD, PSKY->SKYD, PSTG->P, SCVL->SHOE, SJW->HTO, VSNTV->VSNT
Rename-map symbols later reassigned to another company: {'PARA': 'Parabolic Technologies, Inc. Class A Common Stock from 2026-08-07'}


Guesses under a reassigned symbol dated after the reassignment: 0
Ticker guesses: 1962 rows with a guess, 1642 of them in the universe after standardization
Guesses changed by the standardization or the maps: 122 rows


,ticker_guess,TickerGuessStd,rows
5,PARA,SKYD,81
1,GOOGL,GOOG,29
4,NWS,NWSA,6
3,MSGS,MSGK,3
0,BK,BNY,1
2,IAC,PPLI,1
6,VOLV B,VOLV-B,1


## 2) One suggested ticker per company

The unit of the review is the company: every distinct standardized `canonical_company` among the direct, brand and executive references (indirect references name an industry or a country, not a company, and are set aside; see 2.3). For each company the rules below are tried in order, and the first one that finds a universe ticker gives the company's `SuggestedTicker`, `HowFound` and `Score`. A suggestion is only a suggestion: no reference gets a ticker before a person has reviewed its company in section 3, and companies without a suggestion are reviewed too, so that "not in the universe" is a recorded decision rather than the absence of one.

| `HowFound` | rule | `Score` |
|---|---|---|
| `ticker+name` | a ticker guess for this company is in the universe and the company name agrees with that ticker's names | token-set similarity, at least 80 |
| `name` | the standardized name is exactly a universe name of one ticker | 100 |
| `ambiguous` | the standardized name is exactly a universe name of several tickers; they are listed in `Candidates`, no suggestion | 100 |
| `executive` | an executive reference names a person in `executives` of exactly one ticker (first and last name, so "Dave Ricks" meets "David A. Ricks" but a surname alone does not) | name agreement; a low score warns of a namesake |
| `ticker only` | a ticker guess is in the universe, but the names do not agree (right company under a wrong ticker, or a name FMP spells very differently) | token-set similarity, below 80 |
| `similarity` | the closest universe name, letter by letter, both cut before the first legal suffix (INC, CORP, PLC, CO, LTD, GROUP, HLDGS, HOLDINGS); at least 85; a tie between tickers is `ambiguous` | letter-by-letter similarity |
| `none` | nothing found | |

Where a company carries several ticker guesses, the most frequent one is tried first.

In [5]:
NAME_AGREE_THRESHOLD = 80
FUZZY_THRESHOLD = 85
NO_NAME = "(NO COMPANY NAME)"
HONORIFICS = {"MR", "MRS", "MS", "DR", "SIR", "CEO", "CHAIRMAN", "FOUNDER", "PRESIDENT"}
SUFFIX_TOKENS = ["INC", "CORP", "PLC", "CO", "LTD", "GROUP", "HLDGS", "HOLDINGS"]

def name_score(std_name, ticker):
    """Best token-set similarity between a standardized name and the standardized universe names of a ticker."""
    return max((fuzz.token_set_ratio(std_name, n) for n in ticker_to_names.get(ticker) or set()), default=0)

def person_tokens(text):
    return [t for t in re.sub(r"[^A-Za-z\s]", " ", str(text)).upper().split() if t not in HONORIFICS and len(t) > 1]

exec_index = {}   # last name -> [(first name, ticker)]
for _, r in executives.iterrows():
    toks = person_tokens(r["name"])
    if len(toks) >= 2:
        exec_index.setdefault(toks[-1], []).append((toks[0], r["ticker"]))

def executive_tickers(mention):
    """Tickers with an executive whose last name is in the mention, preceded by a compatible first name."""
    toks = person_tokens(mention)
    found = set()
    for i, last in enumerate(toks):
        for first, ticker in exec_index.get(last, []):
            if any(f == first or (len(f) >= 3 and len(first) >= 3 and (f.startswith(first[:3]) or first.startswith(f[:3]))) for f in toks[:i]):
                found.add(ticker)
    return found

def cut_before_suffix(name):
    """The name up to the first legal-entity token; the full name if none occurs."""
    starts = [m.start() for t in SUFFIX_TOKENS for m in [re.search(rf"\b{t}\b", name)] if m]
    cut = name[:min(starts)].strip() if starts else name
    return cut or name

prefix_to_tickers = {}
for n, t in name_to_tickers.items():
    prefix_to_tickers.setdefault(cut_before_suffix(n), set()).update(t)
universe_prefixes = list(prefix_to_tickers)

def suggest(std_name, rows):
    """The first rule that finds a universe ticker for one company (all its references in `rows`)."""
    out = dict(SuggestedTicker=None, HowFound="none", Score=np.nan, Candidates=None)
    guesses = [g for g in rows["TickerGuessStd"].dropna().value_counts().index if g in universe]
    scored = [(g, name_score(std_name, g)) for g in guesses] if std_name else [(g, 0) for g in guesses]
    agree = [(g, s) for g, s in scored if s >= NAME_AGREE_THRESHOLD]
    if agree:
        return {**out, "SuggestedTicker": agree[0][0], "HowFound": "ticker+name", "Score": agree[0][1]}
    hits = name_to_tickers.get(std_name) if std_name else None
    if hits and len(hits) == 1:
        return {**out, "SuggestedTicker": next(iter(hits)), "HowFound": "name", "Score": 100}
    if hits:
        return {**out, "HowFound": "ambiguous", "Score": 100, "Candidates": ", ".join(sorted(hits))}
    exec_hits = set().union(set(), *rows.loc[rows["reference_type"] == "executive", "mention_text"].map(executive_tickers))
    if len(exec_hits) == 1:
        t = next(iter(exec_hits))
        return {**out, "SuggestedTicker": t, "HowFound": "executive", "Score": name_score(std_name, t) if std_name else np.nan}
    if scored:
        return {**out, "SuggestedTicker": scored[0][0], "HowFound": "ticker only", "Score": scored[0][1]}
    if std_name:
        hits = [(n, s) for n, s, _ in process.extract(cut_before_suffix(std_name), universe_prefixes, scorer=fuzz.ratio, limit=5) if s >= FUZZY_THRESHOLD]
        if hits:
            top = hits[0][1]
            tickers = sorted(set().union(*(prefix_to_tickers[n] for n, s in hits if s == top)))
            if len(tickers) == 1:
                return {**out, "SuggestedTicker": tickers[0], "HowFound": "similarity", "Score": top}
            return {**out, "HowFound": "ambiguous", "Score": top, "Candidates": ", ".join(tickers)}
    return out

### 2.1) The company list

All direct, brand and executive references grouped by standardized company name, with what the reviewer needs to judge the company: the LLM's spelling, how often and when it is mentioned, every mention with its count, the ticker guesses, and one example post. Then the rules of section 2 give the suggestion.

In [6]:
HOW_FOUND_ORDER = ["ticker+name", "name", "executive", "ticker only", "similarity", "ambiguous", "none"]

is_indirect = refs["reference_type"].eq("indirect")
refs["CompanyKey"] = np.where(is_indirect, None, refs["CompanyStd"].replace("", NO_NAME))
company_rows = refs[~is_indirect]

def counts_text(s, cap=400):
    """'value (count) | value (count) | ...' in descending count order, cut at `cap` characters."""
    return " | ".join(f"{v} ({n})" for v, n in s.dropna().astype(str).value_counts().items())[:cap]

review_companies = []
for key, rows in company_rows.groupby("CompanyKey", sort=False):
    first = rows.sort_values("created_at").iloc[0]
    review_companies.append({
        "CompanyKey": key,
        "Company": rows["canonical_company"].mode().iat[0] if rows["canonical_company"].notna().any() else NO_NAME.lower(),
        "References": len(rows), "Posts": rows["ID"].nunique(),
        "FirstPost": rows["post_date"].min(), "LastPost": rows["post_date"].max(),
        "Mentions": counts_text(rows["mention_text"]),
        "TickerGuess": counts_text(rows["TickerGuessStd"], cap=100),
        "ExamplePost": re.sub(r"\s+", " ", str(first["text"]))[:300],
        **suggest("" if key == NO_NAME else key, rows),
    })
review_companies = pd.DataFrame(review_companies)
review_companies["SuggestedCompany"] = review_companies["SuggestedTicker"].map(ticker_to_fmp)
review_companies["Score"] = review_companies["Score"].round(0)

# order of the review sheet: companies with a suggestion or candidates first, then the rest; most-mentioned first within each
review_companies["has_suggestion"] = review_companies["SuggestedTicker"].notna() | review_companies["Candidates"].notna()
review_companies = review_companies.sort_values(["has_suggestion", "References", "CompanyKey"], ascending=[False, False, True]).reset_index(drop=True)

suggestion_summary = (review_companies.groupby("HowFound").agg(companies=("CompanyKey", "size"), references=("References", "sum"))
                      .reindex(HOW_FOUND_ORDER).fillna(0).astype(int))
print(f"{len(review_companies)} companies ({int(review_companies['References'].sum())} references) to review; "
      f"{int(review_companies['SuggestedTicker'].notna().sum())} with a suggested universe ticker")
display(suggestion_summary)
display(review_companies.loc[review_companies["SuggestedTicker"].notna(), ["Company", "References", "TickerGuess", "SuggestedTicker", "SuggestedCompany", "HowFound", "Score"]].head(15))

559 companies (2819 references) to review; 157 with a suggested universe ticker


,companies,references
HowFound,,
ticker+name,138,1562
name,5,16
executive,1,1
ticker only,3,89
similarity,10,10
ambiguous,0,0
none,402,1141


,Company,References,TickerGuess,SuggestedTicker,SuggestedCompany,HowFound,Score
0,Fox Corporation,337,FOXA (337),FOXA,Fox Corporation,ticker+name,100.0
1,News Corporation,121,NWSA (119) | NEWS (2),NWSA,News Corporation,ticker+name,100.0
2,"Tesla, Inc.",121,TSLA (121),TSLA,"Tesla, Inc.",ticker+name,100.0
3,Comcast Corporation,97,CMCSA (97),CMCSA,Comcast Corporation,ticker+name,100.0
4,"Warner Bros. Discovery, Inc.",95,WBD (95),WBD,"Warner Bros. Discovery, Inc.",ticker+name,100.0
5,The Walt Disney Company,83,DIS (83),DIS,The Walt Disney Company,ticker+name,100.0
6,Paramount Global,78,SKYD (76),SKYD,Skydance Corporation,ticker only,72.0
7,Apple Inc.,55,AAPL (55),AAPL,Apple Inc.,ticker+name,100.0
8,The Boeing Company,39,BA (39),BA,The Boeing Company,ticker+name,100.0
9,NVIDIA Corporation,31,NVDA (31),NVDA,NVIDIA Corporation,ticker+name,100.0


### 2.2) Guessed tickers outside the universe

A check on the maps of 1.3: every guessed ticker that is not in the universe, with its company. One that is a share class or a rename of a universe ticker belongs in the share-class or rename map; the rest are companies he did not trade (Toyota, The New York Times Company, Trump Media & Technology Group), which the review confirms one by one.

In [7]:
not_in_universe = (company_rows[company_rows["TickerGuessStd"].notna() & ~company_rows["TickerGuessStd"].isin(universe)]
                   .groupby(["TickerGuessStd", "canonical_company"]).size().reset_index(name="references").sort_values("references", ascending=False))
print(f"Guessed tickers outside the universe: {not_in_universe['TickerGuessStd'].nunique()} distinct; the 25 most frequent:")
display(not_in_universe.head(25))

Guessed tickers outside the universe: 48 distinct; the 25 most frequent:


,TickerGuessStd,canonical_company,references
27,NYT,The New York Times Company,107
9,EDR,"Endeavor Group Holdings, Inc.",51
7,DJT,Trump Media & Technology Group Corp.,25
36,TM,Toyota Motor Corporation,14
52,X,United States Steel Corporation,10
26,NXST,"Nexstar Media Group, Inc.",7
8,DMGT,Daily Mail and General Trust plc,6
16,HMC,"Honda Motor Co., Ltd.",6
24,NVO,Novo Nordisk A/S,6
15,GCI,"Gannett Co., Inc.",6


### 2.3) Indirect references

Industry, commodity, group-of-companies and country statements have no company to map. They keep `mention_text` as the theme and `ticker = NULL`, with `status = indirect`, for the descriptive analysis and for the robustness check that expands industries to the universe's `sector` / `industry`. They are not part of the review.

In [8]:
print(f"Indirect references: {int(is_indirect.sum())} rows in {refs.loc[is_indirect, 'ID'].nunique()} posts; "
      f"company references for the review: {len(company_rows)} rows in {company_rows['ID'].nunique()} posts")

Indirect references: 3649 rows in 2395 posts; company references for the review: 2819 rows in 2248 posts


## 3) Review of every company

Nothing gets a universe ticker in the final dataset without a recorded decision (`tasks/lessons.md`, 2026-11-03: a rule is a proposal, and every company–ticker link gets a review decision). The decisions sit in `TickerTwitterReview.xlsx`, sheets in the order they are applied:

| Sheet | Written by | Content |
|---|---|---|
| `ReadMe` | this notebook | how to fill in the workbook, and the review status |
| `1_CompanyReview` | this notebook + the reviewer | one row per company of section 2 with its suggestion; the yellow columns `Review`, `TickerOverride`, `Reason` are the reviewer's; `TickerFinal` shows the outcome |
| `2_MentionExceptions` | the reviewer (+ `References`, counted here) | exceptions inside a company, by mention and optionally by date: Tylenol under Johnson & Johnson is Kenvue; MSNBC and CNBC under Comcast are Versant after the spin-off; a naming-rights arena is no reference to the bank |
| `3_NotListedOnPostDate` | this notebook | references that lose their ticker because the company had no price on the post date (3.4) |
| `Entities` | `4) CompanyDB/BuildEntities.py` | identity events of the universe companies (Step 4.2), not used here |

**How to decide a company in `1_CompanyReview`** (the same convention as `2025ManuallyReviewed` in `2) Trades/TickerReview.xlsx`). The question is *which stock is this company*, for every company, traded or not, because Part 1 needs all listed companies he posted about:

* `TickerOverride` = the company's stock ticker, on whichever exchange it is listed (`NYT`, `TM` for Toyota's ADR, `DJT`), whether or not he traded it. A ticker in the trade universe makes the company's references join the trades; any other ticker marks the company as listed but not traded.
* `TickerOverride` = `NONE`: the company had no public listing at any time in the sample window (a private company, a government body, a brand that is not a company, or a company delisted before 2025-01-20). A company listed for part of the window keeps its ticker; the dates are handled per reference in 3.4.
* `Review = v` alone: the `SuggestedTicker` is right. `v` together with an override means the same as the override (the reviewer has looked, and the override is the decision).
* `Reason`: required with an override.
* All empty: pending. The company's references get no ticker until it is decided, and the notebook lists it.

The review was completed by the authors on 2026-11-08 (every company decided). The suggestions and a first pass over the 159 companies with a suggestion were prepared automatically on 2026-11-06 and 2026-11-07; the authors checked and, where needed, changed them.

A rerun regenerates every other column and keeps the reviewer's columns by `CompanyKey`, the standardized company name. A decision whose company has disappeared from the data (after a change of the labels) is kept at the bottom of the sheet with `References = 0`, so that no decision is ever lost silently.

### 3.1) Reading and checking the company decisions

The notebook stops on a `Review` other than `v`, on an override that is not a ticker-shaped string or `NONE`, and on an override without a `Reason`. An override ticker under an old symbol follows the rename chain of 1.3. Each company gets a `TickerFinal` (a ticker or `NONE`) and a `Decision`: `accepted` (the suggestion), `corrected` (a universe ticker set by the reviewer), `listed_not_traded` (a ticker outside the universe), `not_listed` (`NONE`), `pending`.

In [9]:
COMPANY_SHEET, EXCEPTION_SHEET, UNLISTED_SHEET = "1_CompanyReview", "2_MentionExceptions", "3_NotListedOnPostDate"
HAND_COLS = ["Review", "TickerOverride", "Reason"]
EXCEPTION_COLS = ["Company", "Mention", "FromDate", "ToDate", "Ticker", "Reason"]
TICKER_SHAPE = r"^[A-Z0-9][A-Z0-9.\-^]{0,11}$"   # any exchange's symbol: SKYD, BRK-B, 5401.T, WRB^E

def as_ticker(t):
    """NONE, or the standardized ticker (share class and rename chain of 1.3)."""
    if not isinstance(t, str):
        return t
    return "NONE" if t.strip().upper() == "NONE" else standardize_ticker(t)

def read_sheet(name, columns):
    """A sheet of the review workbook with all cells as stripped text; an empty frame when the workbook or sheet does not exist yet."""
    if not os.path.exists(review_path) or name not in load_workbook(review_path, read_only=True).sheetnames:
        return pd.DataFrame(columns=columns)
    df = pd.read_excel(review_path, sheet_name=name, dtype=str).dropna(how="all")
    missing = set(columns) - set(df.columns)
    assert not missing, f"sheet {name} lacks the columns {missing}"
    return df.apply(lambda s: s.str.strip()).replace("", np.nan)

hand = read_sheet(COMPANY_SHEET, ["CompanyKey", "Company"] + HAND_COLS)
hand = hand[hand[HAND_COLS].notna().any(axis=1)][["CompanyKey", "Company"] + HAND_COLS]
assert not hand["CompanyKey"].duplicated().any(), f"a company has more than one row: {hand.loc[hand['CompanyKey'].duplicated(), 'CompanyKey'].tolist()}"

hand["Review"] = hand["Review"].str.lower()
bad_review = hand[hand["Review"].notna() & (hand["Review"] != "v")]
assert bad_review.empty, f"Review must be 'v' or empty: {bad_review[['CompanyKey', 'Review']].values.tolist()}"
hand["TickerOverride"] = hand["TickerOverride"].map(as_ticker)
bad_ticker = hand[hand["TickerOverride"].notna() & (hand["TickerOverride"] != "NONE") & ~hand["TickerOverride"].str.match(TICKER_SHAPE)]
assert bad_ticker.empty, f"TickerOverride is neither a ticker nor NONE: {bad_ticker[['CompanyKey', 'TickerOverride']].values.tolist()}"
no_reason = hand[hand["TickerOverride"].notna() & hand["Reason"].isna()]
assert no_reason.empty, f"a TickerOverride without a Reason: {no_reason['CompanyKey'].tolist()}"

review = review_companies.merge(hand[["CompanyKey"] + HAND_COLS], on="CompanyKey", how="left")
# decisions whose company is no longer in the data: kept at the bottom, References = 0
stale = hand[~hand["CompanyKey"].isin(review_companies["CompanyKey"])].copy()
if len(stale):
    stale["References"] = 0
    review = pd.concat([review, stale[["CompanyKey", "Company", "References"] + HAND_COLS]], ignore_index=True)

def final_ticker(r):
    if isinstance(r["TickerOverride"], str):
        return r["TickerOverride"]
    if r["Review"] == "v":
        return r["SuggestedTicker"] if isinstance(r["SuggestedTicker"], str) else "NONE"
    return None   # pending

def decision(r):
    t = r["TickerFinal"]
    if t is None:
        return "pending"
    if t == "NONE":
        return "not_listed"
    if t not in universe:
        return "listed_not_traded"
    return "accepted" if t == r["SuggestedTicker"] else "corrected"

DECISION_ORDER = ["accepted", "corrected", "listed_not_traded", "not_listed", "pending"]
review["TickerFinal"] = review.apply(final_ticker, axis=1)
review["Decision"] = review.apply(decision, axis=1)
live = review["References"] > 0
print(f"Companies: {int(live.sum())}; decided {int((live & (review['Decision'] != 'pending')).sum())}, pending {int((live & (review['Decision'] == 'pending')).sum())}")
print(f"Decisions kept for companies no longer in the data: {len(stale)}", stale["CompanyKey"].tolist())
display(review[live].groupby(["HowFound", "Decision"]).agg(companies=("CompanyKey", "size"), references=("References", "sum")).unstack(fill_value=0))
outside = review[live & (review["Decision"] == "listed_not_traded")]
print(f"Listed companies outside the trade universe: {outside['TickerFinal'].nunique()} tickers over {len(outside)} companies:", ", ".join(sorted(outside["TickerFinal"].unique())))

Companies: 559; decided 559, pending 0
Decisions kept for companies no longer in the data: 0 []


companies                                        references                                       
Decision     accepted corrected listed_not_traded not_listed   accepted corrected listed_not_traded not_listed
HowFound                                                                                                      
executive           0         0                 0          1          0         0                 0          1
name                3         0                 0          2         12         0                 0          4
none                0         6                40        356          0        80               258        803
similarity          3         0                 0          7          3         0                 0          7
ticker only         2         0                 0          1         83         0                 0          6
ticker+name       133         0                 0          5       1553         0                 0          9

Listed companies outside the trade universe: 36 tickers over 40 companies: AZN, BH, BUD, CRCL, DJT, FEAM, FHI, GEMI, GETY, GSK, HMC, KODK, METC, NMAX, NVO, NVS, NXST, NYT, PHG, RIO, RUM, SAP, SMG, SOBO, SOC, SPOT, SSP, TDAY, TM, TRI, TSM, TTE, UP, VG, WWR, X


### 3.2) Reading and checking the mention exceptions

An exception changes the references of one company whose `mention_text` contains `Mention` (not case-sensitive; several alternatives separated by `|`, as in `MSNBC | MSDNC | CNBC`), optionally only for posts dated from `FromDate` to `ToDate` (both inclusive). `Ticker` is the stock ticker the matched mentions belong to (as in `1_CompanyReview`, any exchange) or `NONE`, and `Reason` is required. `Company` is matched after the standardization of 1.2, so any spelling of the company works. The notebook stops on an exception that matches no reference, and on a reference that two exceptions match.

In [10]:
exc = read_sheet(EXCEPTION_SHEET, EXCEPTION_COLS)
exc = exc[exc[["Company", "Mention", "Ticker"]].notna().any(axis=1)].reset_index(drop=True)
assert exc[["Company", "Mention", "Ticker", "Reason"]].notna().all().all(), "every exception needs Company, Mention, Ticker and Reason"
exc["CompanyKey"] = exc["Company"].apply(standardize_company)
exc["Ticker"] = exc["Ticker"].map(as_ticker)
bad = exc[(exc["Ticker"] != "NONE") & ~exc["Ticker"].str.match(TICKER_SHAPE)]
assert bad.empty, f"exception Ticker is neither a ticker nor NONE: {bad[['Company', 'Mention', 'Ticker']].values.tolist()}"
for c in ["FromDate", "ToDate"]:
    exc[c] = exc[c].str[:10]
    assert exc[c].dropna().str.match(r"^\d{4}-\d{2}-\d{2}$").all(), f"{c} must be a date YYYY-MM-DD"

exception_of = pd.Series(np.nan, index=refs.index)   # row number in exc
for i, e in exc.iterrows():
    alternatives = [a.strip().lower() for a in e["Mention"].split("|") if a.strip()]
    mention = refs["mention_text"].fillna("").str.lower()
    hit = (refs["CompanyKey"] == e["CompanyKey"]) & mention.map(lambda m: any(a in m for a in alternatives))
    if isinstance(e["FromDate"], str):
        hit &= refs["post_date"] >= e["FromDate"]
    if isinstance(e["ToDate"], str):
        hit &= refs["post_date"] <= e["ToDate"]
    assert hit.any(), f"exception matches no reference: {e[['Company', 'Mention', 'FromDate', 'ToDate']].tolist()}"
    twice = hit & exception_of.notna()
    assert not twice.any(), f"references matched by two exceptions (rows {i + 2} and others): {refs.loc[twice, ['ID', 'mention_text']].values.tolist()}"
    exception_of[hit] = i
exc["References"] = exception_of.value_counts().reindex(exc.index).fillna(0).astype(int)
print(f"Mention exceptions: {len(exc)} rules covering {int(exc['References'].sum())} references")
display(exc[["Company", "Mention", "FromDate", "ToDate", "Ticker", "References"]])

Mention exceptions: 11 rules covering 30 references


,Company,Mention,FromDate,ToDate,Ticker,References
0,Johnson & Johnson,Tylenol,NaN,NaN,KVUE,6
1,Comcast Corporation,MSNBC | MSDNC | MS NOW | MSNOW | CNBC,2026-01-01,NaN,VSNT,12
2,"Warner Bros. Discovery, Inc.",FoxNews,NaN,NaN,FOXA,1
3,"Warner Bros. Discovery, Inc.",Game of Thrones,2026-05-16,2026-05-16,NONE,1
4,The Walt Disney Company,The Force Awakens,2026-05-16,2026-05-16,NONE,1
5,The Walt Disney Company,A New Hope,2026-05-04,2026-05-04,NONE,1
6,The Walt Disney Company,The Incredible Hulk,2026-08-19,2026-08-19,NONE,1
7,Fox Corporation,FOX5Vegas,NaN,NaN,NONE,2
8,The Coca-Cola Company,Coca-Cola 600,NaN,NaN,NONE,2
9,General Motors Company,Cadillac Championship,NaN,NaN,NONE,2


### 3.3) Applying the decisions

Every company reference gets its company's `TickerFinal` as `public_ticker`; a pending company gives none. A mention exception then overrules the company decision for the references it matches. From `public_ticker` follow `ticker` (the same symbol when it is in the trade universe, the join key with the trades) and `status`, one word per reference: `universe` (a trade-universe ticker), `listed_not_traded` (a stock he never traded), `not_listed` (no listing in the window), `indirect` (an industry or country, no company), `pending`. The reviewer's reason comes along as `reason`.

In [11]:
r_by_key = review[live].set_index("CompanyKey")
on_company = refs["CompanyKey"].notna()
refs["company_ticker"] = refs["CompanyKey"].map(r_by_key["TickerFinal"])   # the decision: a ticker, NONE, or None while pending
refs["reason"] = refs["CompanyKey"].map(r_by_key["Reason"])
refs["decided_by"] = np.where(on_company, "company review", "no company (indirect reference)")

has_exc = exception_of.notna()
e_rows = exc.loc[exception_of[has_exc].astype(int)]
refs.loc[has_exc, "company_ticker"] = e_rows["Ticker"].values
refs.loc[has_exc, "reason"] = e_rows["Reason"].values
refs.loc[has_exc, "decided_by"] = "mention exception"

STATUS_ORDER = ["universe", "listed_not_traded", "not_listed", "not_listed_on_post_date", "indirect", "pending"]
def status_of(t, on_company):
    if not on_company:
        return "indirect"
    if t is None or (isinstance(t, float) and np.isnan(t)):
        return "pending"
    if t == "NONE":
        return "not_listed"
    return "universe" if t in universe else "listed_not_traded"

refs["public_ticker"] = refs["company_ticker"].where(refs["company_ticker"].notna() & (refs["company_ticker"] != "NONE"), None)
refs["ticker"] = refs["public_ticker"].where(refs["public_ticker"].isin(universe), None)
refs["status"] = [status_of(t, c) for t, c in zip(refs["company_ticker"], on_company)]
display(refs.groupby("status").agg(references=("ref_id", "size"), posts=("ID", "nunique"), with_public_ticker=("public_ticker", "count"), with_universe_ticker=("ticker", "count")).reindex([s for s in STATUS_ORDER if s != "not_listed_on_post_date"]))

,references,posts,with_public_ticker,with_universe_ticker
status,,,,
universe,1720.0,1454.0,1720.0,1720.0
listed_not_traded,258.0,249.0,258.0,0.0
not_listed,841.0,725.0,0.0,0.0
indirect,3649.0,2395.0,0.0,0.0
pending,NaN,NaN,NaN,NaN


### 3.4) Listing check: no ticker on a day the company was not listed

A reference whose post date falls outside the price series of its ticker (`3) Prices/1) FetchPrices/prices.csv`) loses the ticker and gets `status = not_listed_on_post_date`: the company was not listed that day, so there is no trade or return to link it to (SpaceX before its listing on 2026-06-12; WK Kellogg after its takeover, series ends 2025-09-25). The company itself stays a listed company in the review and in the Part 1 share. These references are listed in `3_NotListedOnPostDate`. The check needs a price series, which only universe tickers have; for a listed company outside the universe (Newsmax listed 2025-03-31, U.S. Steel delisted 2025-06-18) the reviewer's `Reason` records the dates, and the references keep `public_ticker`.

In [12]:
prices = pd.read_csv(PRICES_CSV, usecols=["date", "ticker"])
series = prices.groupby("ticker")["date"].agg(first_price="min", last_price="max")
lo, hi = refs["ticker"].map(series["first_price"]), refs["ticker"].map(series["last_price"])
unlisted = refs["ticker"].notna() & lo.notna() & ((refs["post_date"] < lo) | (refs["post_date"] > hi))

not_listed = (refs[unlisted].assign(PriceSeriesFrom=lo[unlisted], PriceSeriesTo=hi[unlisted])
              .groupby(["canonical_company", "ticker", "PriceSeriesFrom", "PriceSeriesTo"])
              .agg(References=("ref_id", "size"), FirstPost=("post_date", "min"), LastPost=("post_date", "max"),
                   Mentions=("mention_text", counts_text))
              .reset_index().rename(columns={"canonical_company": "Company", "ticker": "Ticker"}))
refs.loc[unlisted, "reason"] = ("not listed on the post date: the " + refs.loc[unlisted, "ticker"] + " price series runs "
                                + lo[unlisted] + " to " + hi[unlisted] + " (prices.csv)")
refs.loc[unlisted, ["ticker", "public_ticker"]] = None
refs.loc[unlisted, "status"] = "not_listed_on_post_date"
refs.loc[unlisted, "decided_by"] = "listing check (prices.csv)"
no_series = sorted(set(refs.loc[refs["ticker"].notna() & ~refs["ticker"].isin(series.index), "ticker"]))
print(f"Not listed on the post date: {int(unlisted.sum())} references")
display(not_listed)
print(f"Universe tickers without a price series (kept, no listing check possible): {no_series}")

Not listed on the post date: 29 references


,Company,Ticker,PriceSeriesFrom,PriceSeriesTo,References,FirstPost,LastPost,Mentions
0,Space Exploration Technologies Corp.,SPCX,2026-06-12,2026-08-31,7,2025-03-09,2025-08-22,"@SpaceX (4) | Starlink (1) | SpaceX #Crew9 (1) | Go Falcon, Go Dragon, Go Cr..."
1,SpaceX,SPCX,2026-06-12,2026-08-31,4,2025-01-29,2025-03-17,@SpaceX (2) | members of @elonmusk’s SpaceX team (1) | SpaceX Dragon (1)
2,WK Kellogg Co,KLG,2025-01-10,2025-09-25,1,2026-08-11,2026-08-11,Kellogg (1)
3,X Corp,SPCX,2026-06-12,2026-08-31,3,2025-04-12,2026-03-20,X (2) | LITTLE X (1)
4,X Corp.,SPCX,2026-06-12,2026-08-31,14,2025-01-26,2026-05-13,X (9) | President Donald J. Trump’s X account (1) | little X (1) | X! (1) | ...


Universe tickers without a price series (kept, no listing check possible): ['HOG']


### 3.5) Writing the review workbook

The notebook rewrites `ReadMe`, `1_CompanyReview` (keeping the reviewer's columns), `2_MentionExceptions` (keeping everything but the `References` count) and `3_NotListedOnPostDate`. Every other sheet, such as `Entities`, is left as it is. The reviewer's columns are yellow and `TickerFinal` is grey; the header row is frozen and filtered.

In [13]:
HEADER_FILL = PatternFill("solid", fgColor="1F4E78")
HAND_FILL = PatternFill("solid", fgColor="FFF2CC")
RESULT_FILL = PatternFill("solid", fgColor="E7E6E6")
WIDTHS = {"Company": 34, "Mentions": 50, "ExamplePost": 60, "SuggestedCompany": 30, "Reason": 50, "ReviewedBy": 22, "Explanation": 120,
          "Item": 26, "CompanyKey": 30, "TickerGuess": 16, "Candidates": 18, "Mention": 22}

def write_sheet(wb, name, df, hand_cols=(), result_cols=(), position=None):
    if name in wb.sheetnames:
        del wb[name]
    ws = wb.create_sheet(name, position)
    ws.append(list(df.columns))
    for row in df.itertuples(index=False):
        ws.append([None if (isinstance(v, float) and np.isnan(v)) else v for v in row])
    for j, col in enumerate(df.columns, start=1):
        letter = ws.cell(1, j).column_letter
        ws.cell(1, j).font = Font(bold=True, color="FFFFFF")
        ws.cell(1, j).fill = HEADER_FILL
        ws.cell(1, j).alignment = Alignment(vertical="center")
        ws.column_dimensions[letter].width = WIDTHS.get(col, max(10, min(len(col) + 4, 20)))
        fill = HAND_FILL if col in hand_cols else RESULT_FILL if col in result_cols else None
        if fill:
            for i in range(2, max(len(df) + 2, 200)):
                ws.cell(i, j).fill = fill
    ws.freeze_panes = "B2"
    ws.auto_filter.ref = ws.dimensions
    return ws

SHEET_COLS = ["Company", "References", "Posts", "FirstPost", "LastPost", "Mentions", "TickerGuess", "SuggestedTicker", "SuggestedCompany",
              "HowFound", "Score", "Candidates", "ExamplePost", "Review", "TickerOverride", "Reason", "TickerFinal", "Decision", "CompanyKey"]
n_live, n_pending = int(live.sum()), int((live & (review["Decision"] == "pending")).sum())
dec_counts = review[live]["Decision"].value_counts().reindex(DECISION_ORDER).fillna(0).astype(int)
readme = pd.DataFrame([
    ("Status of the review", f"{n_live - n_pending} of {n_live} companies decided, {n_pending} pending; {len(exc)} mention exceptions. "
     f"Decisions: {dec_counts['accepted']} suggestions accepted, {dec_counts['corrected']} universe tickers set by the reviewer, "
     f"{dec_counts['listed_not_traded']} listed companies outside the trade universe, {dec_counts['not_listed']} not listed (NONE). "
     "Reviewed by the authors, completed 2026-11-08; suggestions and a first pass over the companies with a suggestion prepared automatically, 2026-11-06/07."),
    ("What this workbook is", "Every manual decision of Step 5.2: which stock each company in Trump's posts is, if any. "
     "TwitterTickerCleaning.ipynb (section 3) reads it on every run and applies it; nobody edits the data by hand."),
    ("Why", "Part 1 of the thesis: the share of listed companies he posted about that he also purchased. So every company needs a verdict: "
     "its ticker (any exchange) or NONE. A ticker in the trade universe (company.db, the tickers he traded) also links the posts to the trades for Parts 2 and 3."),
    ("Order of application", "1) 1_CompanyReview decides each company; 2) 2_MentionExceptions overrules it for single mentions or dates; "
     "3) a reference dated outside its ticker's price series loses the ticker (3_NotListedOnPostDate)."),
    ("1_CompanyReview", "One row per company the LLM named (direct, brand and executive references; spellings that differ only in case, "
     "punctuation or 'Corporation'/'Corp' are one company). Rows with a suggestion first, then the rest; most-mentioned first. "
     "Only the yellow columns are yours; the notebook rewrites the others."),
    ("  TickerOverride", "The company's stock ticker on whichever exchange it is listed (SKYD, NYT, TM for Toyota's ADR), traded by him or not; "
     "or NONE when the company had no public listing at any time in the sample window (private, government, not a company, delisted before 2025-01-20). "
     "A company listed for part of the window keeps its ticker; the dates are handled per reference by the notebook. Reason is required."),
    ("  Review = v", "Alone: the SuggestedTicker is right. Together with a TickerOverride: the override is the decision (v only records that the row was looked at)."),
    ("  Reason", "Why; required with an override, welcome otherwise."),
    ("  TickerFinal, Decision (grey)", "Written by the notebook: the ticker the company's references receive (or NONE; empty while pending), and the kind of decision: "
     "accepted (the suggestion), corrected (another universe ticker), listed_not_traded (a ticker outside the universe), not_listed (NONE), pending."),
    ("  HowFound", "How the suggestion was found: ticker+name (guess in the universe and the names agree), name (exact universe name), "
     "executive (a named executive of that company), ticker only (guess in the universe, names differ: check), similarity (closest "
     "universe name: check), ambiguous (several candidates, see Candidates), none (nothing found). Score: 0-100 agreement of the names."),
    ("2_MentionExceptions", "Exceptions inside a company. A row applies to the references of Company whose mention contains Mention "
     "(not case-sensitive; alternatives separated by |, e.g. MSNBC | CNBC), optionally only from FromDate to ToDate (YYYY-MM-DD, inclusive). Ticker: the stock these mentions belong to, or NONE. "
     "Reason required. References is counted by the notebook. Example: Johnson & Johnson / Tylenol -> KVUE."),
    ("3_NotListedOnPostDate", "Generated, for information: references whose ticker had no price on the post date and so get no ticker."),
    ("Entities", "Written by 4) CompanyDB/BuildEntities.py (Step 4.2); not used by this notebook."),
], columns=["Item", "Explanation"])

company_sheet = review.sort_values(["References"], key=lambda s: s.eq(0), kind="stable")[SHEET_COLS]
wb = load_workbook(review_path) if os.path.exists(review_path) else Workbook()
if "Sheet" in wb.sheetnames and len(wb.sheetnames) == 1:
    wb["Sheet"].title = "_tmp"
write_sheet(wb, "ReadMe", readme, position=0)
ws = write_sheet(wb, COMPANY_SHEET, company_sheet, hand_cols=HAND_COLS, result_cols=["TickerFinal", "Decision"], position=1)
dv = DataValidation(type="list", formula1='"v"', allow_blank=True)
ws.add_data_validation(dv)
dv.add(f"N2:N{max(len(company_sheet) + 1, 200)}")
write_sheet(wb, EXCEPTION_SHEET, exc[EXCEPTION_COLS + ["References"]], hand_cols=EXCEPTION_COLS, result_cols=["References"], position=2)
write_sheet(wb, UNLISTED_SHEET, not_listed, position=3)
if "_tmp" in wb.sheetnames:
    del wb["_tmp"]
wb.active = 1
wb.save(review_path)
print(f"{review_path}: sheets {wb.sheetnames}")

TickerTwitterReview.xlsx: sheets ['ReadMe', '1_CompanyReview', '2_MentionExceptions', '3_NotListedOnPostDate', 'Entities']


## 4) Final dataset

One row per post and company reference, `Twitter_tickers_2025_2026.xlsx`, sheet `Twitter_tickers`, with a `ReadMe` sheet that defines every column. The columns come in three groups: the post (Step 1), what the LLM read in it (Step 5.1), and the decision of this step. The decision is five fields: `company` (the name after review), `public_ticker` (the stock, any exchange), `ticker` (the same symbol when it is in the trade universe: the join key with the trades, under the latest symbol as in Step 2.5), the flags `listed`, `in_universe`, `purchased` (the ticker is in `purchases.csv`), and `status` with `decided_by` and `reason` saying where the decision came from. Everything else about the review (suggestion, rule, score) stays in `TickerTwitterReview.xlsx`.

The trade universe holds 26 tickers he only sold (HOG among them), so `in_universe = 1` does not imply `purchased = 1`; Part 1 counts purchases. Before the export we check that every reference from Step 5.1 is present, that every ticker comes from a decision, is in the universe and has a price on its post date, and that the flags agree with the tickers.

In [14]:
purchased_tickers = set(pd.read_csv(PURCHASES_CSV, usecols=["ticker"])["ticker"])
assert purchased_tickers <= universe, f"purchased tickers outside the universe: {sorted(purchased_tickers - universe)}"
print(f"Purchased tickers (purchases.csv): {len(purchased_tickers)} of {len(universe)} universe tickers; sold only: {len(universe - purchased_tickers)}")

refs["company"] = refs["ticker"].map(ticker_to_fmp).where(refs["ticker"].notna(), refs["canonical_company"])
refs["listed"] = refs["public_ticker"].notna().astype(int)
refs["in_universe"] = refs["ticker"].notna().astype(int)
refs["purchased"] = refs["ticker"].isin(purchased_tickers).astype(int)

FINAL_COLS = ["ID", "created_at", "User", "platform", "text",
              "mention_text", "reference_type", "canonical_company", "sentiment", "relevance", "confidence",
              "company", "public_ticker", "ticker", "listed", "in_universe", "purchased", "status", "decided_by", "reason"]
COLUMN_DOC = [
    ("ID", "the post", "post ID (Step 1); several rows when a post refers to several companies"),
    ("created_at", "the post", "post time, UTC"),
    ("User", "the post", "account (realDonaldTrump, WhiteHouse, ...)"),
    ("platform", "the post", "truthsocial or twitter"),
    ("text", "the post", "the post text as labelled"),
    ("mention_text", "what the LLM read (Step 5.1)", "the words in the post that refer to the company (or the theme of an indirect reference)"),
    ("reference_type", "what the LLM read (Step 5.1)", "direct (name or ticker), brand (product or subsidiary), executive (a named person), indirect (industry, commodity, country: no company)"),
    ("canonical_company", "what the LLM read (Step 5.1)", "the company's legal name as the LLM wrote it; the review groups spellings of one company"),
    ("sentiment", "what the LLM read (Step 5.1)", "sentiment toward the company, -1 to 1"),
    ("relevance", "what the LLM read (Step 5.1)", "how central the company is to the post, 0 to 1"),
    ("confidence", "what the LLM read (Step 5.1)", "the LLM's confidence in the label, 0 to 1"),
    ("company", "the decision (Step 5.2)", "the company after review: the FMP name of the universe ticker, else the LLM's name"),
    ("public_ticker", "the decision (Step 5.2)", "the company's stock ticker on whichever exchange it is listed, from the review; empty when not listed, indirect, or not listed on the post date"),
    ("ticker", "the decision (Step 5.2)", "public_ticker when it is in the trade universe (company.db): the join key with Trump_trades_2025_2026.xlsx, latest symbol; empty otherwise"),
    ("listed", "the decision (Step 5.2)", "1 if public_ticker is set"),
    ("in_universe", "the decision (Step 5.2)", "1 if ticker is set (a company he traded, bought or sold)"),
    ("purchased", "the decision (Step 5.2)", "1 if ticker is in purchases.csv (a company he bought); the Part 1 numerator"),
    ("status", "the decision (Step 5.2)", "universe: carries a trade-universe ticker | listed_not_traded: a listed company he never traded | not_listed: no public listing in the window (NONE in the review) | "
     "not_listed_on_post_date: the company has a ticker but no price on this day (before listing or after delisting) | indirect: no company | pending: not yet reviewed"),
    ("decided_by", "the decision (Step 5.2)", "company review (1_CompanyReview) | mention exception (2_MentionExceptions) | listing check (prices.csv) | no company (indirect reference)"),
    ("reason", "the decision (Step 5.2)", "the reviewer's reason, where one was written, or the listing check's"),
]
Twitter_tickers = refs.sort_values(["created_at", "ID", "ref_id"])[FINAL_COLS].reset_index(drop=True)

with sqlite3.connect(SOCIAL_DB) as conn:
    n_refs_db, n_posts_db = conn.execute("SELECT count(*), count(DISTINCT ID) FROM SocialMediaKeywords").fetchone()
with_ticker = Twitter_tickers["ticker"].notna()
with_public = Twitter_tickers["public_ticker"].notna()
day = Twitter_tickers["created_at"].str[:10]
checks = {
    "every reference from SocialMediaKeywords appears": len(Twitter_tickers) == n_refs_db and Twitter_tickers["ID"].nunique() == n_posts_db,
    "every ticker is a public_ticker in the universe": (Twitter_tickers.loc[with_ticker, "ticker"] == Twitter_tickers.loc[with_ticker, "public_ticker"]).all() and Twitter_tickers.loc[with_ticker, "ticker"].isin(universe).all(),
    "status universe <=> ticker set": ((Twitter_tickers["status"] == "universe") == with_ticker).all(),
    "status listed_not_traded <=> public_ticker set and ticker empty": ((Twitter_tickers["status"] == "listed_not_traded") == (with_public & ~with_ticker)).all(),
    "not_listed, not_listed_on_post_date, indirect and pending rows have no ticker": not Twitter_tickers.loc[Twitter_tickers["status"].isin(["not_listed", "not_listed_on_post_date", "indirect", "pending"]), "public_ticker"].notna().any(),
    "every ticker has a price on its post date (where a series exists)": not (with_ticker & ((day < Twitter_tickers["ticker"].map(series["first_price"])) | (day > Twitter_tickers["ticker"].map(series["last_price"])))).any(),
    "flags agree with the tickers": (Twitter_tickers["listed"] == with_public.astype(int)).all() and (Twitter_tickers["in_universe"] == with_ticker.astype(int)).all() and (Twitter_tickers["purchased"] <= Twitter_tickers["in_universe"]).all(),
}
for k, v in checks.items():
    print(f"{'OK ' if v else 'FAIL'} {k}")
assert all(checks.values())
if (Twitter_tickers["status"] == "pending").any():
    print(f"WARNING: {int((Twitter_tickers['status'] == 'pending').sum())} references belong to companies not yet reviewed")

status_counts = (Twitter_tickers.groupby("status").agg(rows=("ID", "size"), posts=("ID", "nunique"), with_public_ticker=("public_ticker", "count"), with_universe_ticker=("ticker", "count"), purchased=("purchased", "sum"))
                 .reindex(STATUS_ORDER).fillna(0).astype(int))
status_counts["share_of_rows"] = (status_counts["rows"] / len(Twitter_tickers)).round(3)
display(status_counts)

with pd.ExcelWriter("Twitter_tickers_2025_2026.xlsx", engine="openpyxl") as xw:
    Twitter_tickers.to_excel(xw, sheet_name="Twitter_tickers", index=False)
    pd.DataFrame(COLUMN_DOC, columns=["Column", "Group", "Meaning"]).to_excel(xw, sheet_name="ReadMe", index=False)
    ws = xw.sheets["ReadMe"]
    for col, w in zip("ABC", (18, 30, 150)):
        ws.column_dimensions[col].width = w
    for c in ws[1]:
        c.font = Font(bold=True)
print(f"{len(Twitter_tickers)} rows from {Twitter_tickers['ID'].nunique()} posts written to Twitter_tickers_2025_2026.xlsx; "
      f"{int(with_public.sum())} rows with a public ticker ({Twitter_tickers.loc[with_public, 'public_ticker'].nunique()} distinct), "
      f"{int(with_ticker.sum())} with a universe ticker ({Twitter_tickers.loc[with_ticker, 'ticker'].nunique()} distinct), "
      f"{int(Twitter_tickers['purchased'].sum())} on a purchased company ({Twitter_tickers.loc[Twitter_tickers['purchased'] == 1, 'ticker'].nunique()} distinct)")

Purchased tickers (purchases.csv): 1594 of 1620 universe tickers; sold only: 26
OK  every reference from SocialMediaKeywords appears
OK  every ticker is a public_ticker in the universe
OK  status universe <=> ticker set
OK  status listed_not_traded <=> public_ticker set and ticker empty
OK  not_listed, not_listed_on_post_date, indirect and pending rows have no ticker
OK  every ticker has a price on its post date (where a series exists)
OK  flags agree with the tickers


,rows,posts,with_public_ticker,with_universe_ticker,purchased,share_of_rows
status,,,,,,
universe,1691,1438,1691,1691,1689,0.261
listed_not_traded,258,249,258,0,0,0.040
not_listed,841,725,0,0,0,0.130
not_listed_on_post_date,29,29,0,0,0,0.004
indirect,3649,2395,0,0,0,0.564
pending,0,0,0,0,0,0.000


6468 rows from 4376 posts written to Twitter_tickers_2025_2026.xlsx; 1949 rows with a public ticker (178 distinct), 1691 with a universe ticker (142 distinct), 1689 on a purchased company (141 distinct)


## 5) Descriptive analysis

`TwitterDescriptiveAnalysis.xlsx` holds the descriptive tables for the data chapter, one numbered sheet per table in the order the chapter reads them, with a `ReadMe` that defines each one. Counts are numbers and shares are formatted as percentages; every table has a total row where a total makes sense.

| Sheet | Table |
|---|---|
| `1_PostsPerMonth` | all posts per month and account |
| `2_BlockedPosts` | posts the content filter blocked from labelling, per month and account, and their share of the month's posts |
| `3_PostsWithReferences` | per account: labelled posts, and how many refer to anything, to a company, to a listed company, and to a company he purchased |
| `4_ReferenceTypes` | company references per `reference_type`, and how many got a public and a universe ticker |
| `5_Status` | every reference by its `status` (section 3) |
| `6_SuggestionVsReview` | companies: how the suggestion was found (section 2) against what the review decided |
| `7_Part1Share` | the Part 1 number: of the listed companies he posted about, the share he purchased (and the share in the trade universe), over distinct companies, references and posts |
| `8_TopCompanies` | the 50 most-mentioned companies with ticker, listed / traded / purchased and mean sentiment |
| `9_SentimentScores` | the distribution of the sentiment score on company references |
| `10_SentimentClasses` | positive / neutral / negative shares at the pre-registered threshold ±0.5 and at ±0.3 and ±0.7 as sensitivity |

A "company" in tables 7 and 8 is its `public_ticker` where the review gave one (so the two spellings of Paramount, or three Nexstar stations, are one company), else the standardized company name of section 2. Tables 3 to 8 depend on the review: while companies are pending (`1_CompanyReview`), the `ReadMe` says how many, and the numbers are provisional. The review decisions themselves are in `TickerTwitterReview.xlsx` and are not repeated here.

In [15]:
with sqlite3.connect(SOCIAL_DB) as conn:
    posts = pd.read_sql_query("SELECT ID, created_at, User, platform, llm_status FROM SocialMedia", conn)
posts["month"] = pd.to_datetime(posts["created_at"], utc=True, errors="coerce").dt.strftime("%Y-%m")
company_refs = Twitter_tickers[Twitter_tickers["reference_type"] != "indirect"].copy()
company_refs["company_key"] = company_refs["public_ticker"].where(company_refs["public_ticker"].notna(), company_refs["canonical_company"].apply(standardize_company))
ACCOUNTS = sorted(posts["User"].unique())

def with_total(df, label_col, label="Total", sum_cols=None):
    """Appends a total row that sums `sum_cols` (default: every numeric column)."""
    sum_cols = sum_cols or [c for c in df.columns if c != label_col and pd.api.types.is_numeric_dtype(df[c])]
    total = {label_col: label, **{c: df[c].sum() for c in sum_cols}}
    return pd.concat([df, pd.DataFrame([total])], ignore_index=True)

# 1 posts per month and account
t1 = posts.pivot_table(index="month", columns="User", values="ID", aggfunc="count", fill_value=0).reindex(columns=ACCOUNTS, fill_value=0)
t1["All accounts"] = t1.sum(axis=1)
t1 = with_total(t1.reset_index().rename(columns={"month": "Month"}), "Month")

# 2 posts blocked by the content filter, per month and account, and the share of the month's posts
blocked_posts = posts[posts["llm_status"] == "content_filter"]
t2 = blocked_posts.pivot_table(index="month", columns="User", values="ID", aggfunc="count", fill_value=0).reindex(index=t1["Month"][:-1], columns=ACCOUNTS, fill_value=0)
t2["All accounts"] = t2.sum(axis=1)
t2 = with_total(t2.reset_index().rename(columns={"month": "Month"}), "Month")
t2["Share of all posts blocked"] = t2["All accounts"] / t1["All accounts"].values

# 3 posts with references, per account (labelled posts only)
labelled = posts[posts["llm_status"] == "ok"]
sets = {"With any reference": set(Twitter_tickers["ID"]), "With a company reference": set(company_refs["ID"]),
        "With a listed company": set(company_refs.loc[company_refs["listed"] == 1, "ID"]),
        "With a trade-universe company": set(company_refs.loc[company_refs["in_universe"] == 1, "ID"]),
        "With a purchased company": set(company_refs.loc[company_refs["purchased"] == 1, "ID"])}
rows3 = []
for account, g in list(labelled.groupby("User")) + [("All accounts", labelled)]:
    row = {"Account": account, "Labelled posts": len(g)}
    for name, ids in sets.items():
        row[name] = int(g["ID"].isin(ids).sum())
        row[f"% {name[0].lower() + name[1:]}"] = row[name] / len(g)
    rows3.append(row)
t3 = pd.DataFrame(rows3)

# 4 references per reference type
TYPE_ORDER = ["direct", "brand", "executive", "indirect"]
t4 = (Twitter_tickers.groupby("reference_type").agg(References=("ID", "size"), Posts=("ID", "nunique"), **{"With a public ticker": ("listed", "sum"), "With a universe ticker": ("in_universe", "sum"), "Purchased company": ("purchased", "sum")})
      .reindex(TYPE_ORDER).reset_index().rename(columns={"reference_type": "Reference type"}))
t4 = with_total(t4, "Reference type", sum_cols=["References", "With a public ticker", "With a universe ticker", "Purchased company"])
t4.loc[t4.index[-1], "Posts"] = Twitter_tickers["ID"].nunique()
t4["% with a public ticker"] = t4["With a public ticker"] / t4["References"]
t4["% with a universe ticker"] = t4["With a universe ticker"] / t4["References"]

# 5 references by status
STATUS_LABELS = {"universe": "carries a trade-universe ticker (a company he traded)", "listed_not_traded": "a listed company he never traded",
                 "not_listed": "no public listing in the sample window (NONE in the review)", "not_listed_on_post_date": "listed company, but no price on the post date",
                 "indirect": "indirect reference: no company", "pending": "pending: company not yet reviewed"}
t5 = (Twitter_tickers.groupby("status").agg(References=("ID", "size"), Posts=("ID", "nunique"), **{"With a public ticker": ("public_ticker", "count"), "With a universe ticker": ("ticker", "count")})
      .reindex(STATUS_ORDER).fillna(0).astype(int).reset_index())
t5.insert(1, "Meaning", t5["status"].map(STATUS_LABELS))
t5 = with_total(t5.rename(columns={"status": "Status"}), "Status", sum_cols=["References", "With a public ticker", "With a universe ticker"])
t5.loc[t5.index[-1], "Posts"] = Twitter_tickers["ID"].nunique()
t5["% of references"] = t5["References"] / len(Twitter_tickers)

# 6 suggestion against review decision (companies)
t6 = (review[live].pivot_table(index="HowFound", columns="Decision", values="CompanyKey", aggfunc="count", fill_value=0)
      .reindex(index=HOW_FOUND_ORDER, columns=DECISION_ORDER, fill_value=0).fillna(0).astype(int))
t6["Companies"] = t6.sum(axis=1)
t6 = with_total(t6.reset_index().rename(columns={"HowFound": "Suggestion found by"}), "Suggestion found by")

# 7 Part 1: of the listed companies he posted about, the share he purchased (and the share in the trade universe at all)
listed_refs = company_refs[company_refs["listed"] == 1]
def part1_row(label, frame, key):
    base = frame[key].nunique()
    purch = frame.loc[frame["purchased"] == 1, key].nunique()
    uni = frame.loc[frame["in_universe"] == 1, key].nunique()
    return (label, base, purch, purch / base if base else np.nan, uni, uni / base if base else np.nan)
t7 = pd.DataFrame([part1_row("Distinct listed companies posted about", listed_refs, "company_key"),
                   part1_row("References to listed companies", listed_refs.assign(_r=range(len(listed_refs))), "_r"),
                   part1_row("Posts referring to a listed company", listed_refs, "ID")],
                  columns=["Counted over", "Listed companies he posted about", "Purchased (purchases.csv)", "% purchased", "In the trade universe (bought or sold)", "% in the trade universe"])
n_all_companies = company_refs["company_key"].nunique()
t7_note = (f"Denominator: companies with a public ticker in the review (status universe or listed_not_traded); {n_all_companies - listed_refs['company_key'].nunique()} of the {n_all_companies} companies named are not listed and are left out. "
           "Numerator: companies whose ticker is in purchases.csv. A company is its public_ticker.")

# 8 the 50 most-mentioned companies
t8 = (company_refs.groupby("company_key").agg(References=("ID", "size"), Posts=("ID", "nunique"), Ticker=("public_ticker", "first"),
                                              listed=("listed", "max"), in_u=("in_universe", "max"), purch=("purchased", "max"),
                                              name=("company", lambda s: s.mode().iat[0] if s.notna().any() else ""),
                                              **{"Mean sentiment": ("sentiment", "mean")})
      .sort_values(["References", "Posts"], ascending=False).head(50).reset_index(drop=True))
t8.insert(0, "Rank", range(1, len(t8) + 1))
t8.insert(1, "Company", t8["name"])
for col, flag in [("Listed", "listed"), ("In trade universe", "in_u"), ("Purchased", "purch")]:
    t8[col] = np.where(t8[flag] == 1, "yes", "no")
t8 = t8[["Rank", "Company", "Ticker", "Listed", "In trade universe", "Purchased", "References", "Posts", "Mean sentiment"]].round({"Mean sentiment": 2})

# 9 sentiment score distribution
uni_refs = company_refs[company_refs["in_universe"] == 1]
t9 = pd.concat([company_refs["sentiment"].value_counts().rename("Company references"),
                uni_refs["sentiment"].value_counts().rename("In-universe references")], axis=1).fillna(0).astype(int).sort_index()
t9 = t9.reset_index().rename(columns={"index": "Sentiment score", "sentiment": "Sentiment score"})
t9 = with_total(t9, "Sentiment score")
t9.insert(2, "% of company references", t9["Company references"] / len(company_refs))
t9["% of in-universe references"] = t9["In-universe references"] / len(uni_refs)

# 10 sentiment classes at three thresholds
rows10 = []
for thr in (0.5, 0.3, 0.7):
    for cls, f in [("positive", lambda s: s >= thr), ("neutral", lambda s: s.abs() < thr), ("negative", lambda s: s <= -thr)]:
        rows10.append({"Threshold": f"±{thr}" + (" (main)" if thr == 0.5 else " (sensitivity)"), "Class": cls,
                       "Rule": {"positive": f"score ≥ {thr}", "neutral": f"|score| < {thr}", "negative": f"score ≤ -{thr}"}[cls],
                       "% of company references": f(company_refs["sentiment"]).mean(),
                       "% of in-universe references": f(uni_refs["sentiment"]).mean(),
                       "% of all references incl. indirect": f(Twitter_tickers["sentiment"]).mean()})
t10 = pd.DataFrame(rows10)

tables = {
    "1_PostsPerMonth": (t1, "All posts per month and account (labelled or blocked), 2025-01-20 to 2026-08-31."),
    "2_BlockedPosts": (t2, "Posts Azure's content filter blocked, so the LLM gave no label; per month and account, and as a share of the month's posts."),
    "3_PostsWithReferences": (t3, "Labelled posts per account, and how many contain any reference (incl. indirect), a company reference, a listed company, a company in the trade universe, and a company he purchased."),
    "4_ReferenceTypes": (t4, "References per reference_type (direct = name or ticker, brand = product or subsidiary, executive = named person, indirect = industry or country), and how many got a public ticker, a universe ticker, and refer to a purchased company. The total counts distinct posts."),
    "5_Status": (t5, "Every reference by its status in Twitter_tickers_2025_2026.xlsx (the result of the review, the exceptions and the listing check). The total counts distinct posts."),
    "6_SuggestionVsReview": (t6, "Companies (not references): how the rules found a suggestion (section 2) against what the review decided (accepted / corrected / listed_not_traded / not_listed / pending)."),
    "7_Part1Share": (t7, "The Part 1 number: of the listed companies he posted about, the share he purchased, counted over distinct companies, over references and over posts; also the share in the trade universe at all (bought or sold). " + t7_note),
    "8_TopCompanies": (t8, "The 50 most-mentioned companies (direct, brand and executive references), with public ticker, listed / in the trade universe / purchased, and mean sentiment (-1 to 1)."),
    "9_SentimentScores": (t9, "Distribution of the LLM's sentiment score toward the company, on company references and on references with a universe ticker."),
    "10_SentimentClasses": (t10, "Positive / neutral / negative shares at the pre-registered threshold ±0.5 and at ±0.3 and ±0.7 as sensitivity."),
}
status = (f"{n_pending} of {n_live} companies are still pending in the ticker review (1_CompanyReview); their references count as outside the trade universe. "
          "Tables 3-8 are provisional until the review is complete." if n_pending else f"The ticker review is complete: all {n_live} companies decided.")
readme5 = pd.DataFrame([("Status", status), ("Source", "TwitterTickerCleaning.ipynb, section 5; rewritten on every run. Do not edit.")]
                       + [(name, desc) for name, (_, desc) in tables.items()], columns=["Sheet", "Explanation"])

PCT, DEC = "0.0%", "0.00"
wb5 = Workbook()
wb5.remove(wb5.active)
write_sheet(wb5, "ReadMe", readme5)
wb5["ReadMe"].column_dimensions["B"].width = 140
for name, (df, _) in tables.items():
    ws = write_sheet(wb5, name, df)
    for j, col in enumerate(df.columns, start=1):
        fmt = PCT if col.startswith("%") or col.startswith("Share") else DEC if col == "Mean sentiment" else None
        if fmt:
            for i in range(2, len(df) + 2):
                ws.cell(i, j).number_format = fmt
        ws.column_dimensions[ws.cell(1, j).column_letter].width = max(12, min(len(str(col)) + 3, 45))
    if name in {"1_PostsPerMonth", "2_BlockedPosts", "4_ReferenceTypes", "5_Status", "6_SuggestionVsReview", "9_SentimentScores"}:
        for j in range(1, len(df.columns) + 1):
            ws.cell(len(df) + 1, j).font = Font(bold=True)
    ws.auto_filter.ref = None
wb5["8_TopCompanies"].column_dimensions["B"].width = 40
wb5["5_Status"].column_dimensions["B"].width = 50
wb5["7_Part1Share"].column_dimensions["A"].width = 40
wb5.save("TwitterDescriptiveAnalysis.xlsx")
print(f"TwitterDescriptiveAnalysis.xlsx written: {len(wb5.sheetnames)} sheets {wb5.sheetnames}")
print(status)
display(t7)
display(t5)
display(t8.head(15))

TwitterDescriptiveAnalysis.xlsx written: 11 sheets ['ReadMe', '1_PostsPerMonth', '2_BlockedPosts', '3_PostsWithReferences', '4_ReferenceTypes', '5_Status', '6_SuggestionVsReview', '7_Part1Share', '8_TopCompanies', '9_SentimentScores', '10_SentimentClasses']
The ticker review is complete: all 559 companies decided.


,Counted over,Listed companies he posted about,Purchased (purchases.csv),% purchased,In the trade universe (bought or sold),% in the trade universe
0,Distinct listed companies posted about,178,141,0.792135,142,0.797753
1,References to listed companies,1949,1689,0.866598,1691,0.867624
2,Posts referring to a listed company,1620,1436,0.886420,1438,0.887654


,Status,Meaning,References,Posts,With a public ticker,With a universe ticker,% of references
0,universe,carries a trade-universe ticker (a company he traded),1691,1438.0,1691,1691,0.261441
1,listed_not_traded,a listed company he never traded,258,249.0,258,0,0.039889
2,not_listed,no public listing in the sample window (NONE in the review),841,725.0,0,0,0.130025
3,not_listed_on_post_date,"listed company, but no price on the post date",29,29.0,0,0,0.004484
4,indirect,indirect reference: no company,3649,2395.0,0,0,0.564162
5,pending,pending: company not yet reviewed,0,0.0,0,0,0.000000
6,Total,NaN,6468,4376.0,1949,1691,1.000000


,Rank,Company,Ticker,Listed,In trade universe,Purchased,References,Posts,Mean sentiment
0,1,Fox Corporation,FOXA,yes,yes,yes,336,335,0.10
1,2,"Tesla, Inc.",TSLA,yes,yes,yes,121,121,0.28
2,3,News Corporation,NWSA,yes,yes,yes,121,118,-0.29
3,4,The New York Times Company,NYT,yes,no,no,107,107,-0.82
4,5,"Warner Bros. Discovery, Inc.",WBD,yes,yes,yes,93,93,-0.71
5,6,Comcast Corporation,CMCSA,yes,yes,yes,85,85,-0.44
6,7,Skydance Corporation,SKYD,yes,yes,yes,83,82,-0.30
7,8,"TKO Group Holdings, Inc.",TKO,yes,yes,yes,80,80,0.28
8,9,The Walt Disney Company,DIS,yes,yes,yes,80,78,-0.46
9,10,The Trump Organization,NaN,no,no,no,63,63,0.27


→ We now have one row per post and company, each company classified as listed or not, in the trade universe or not, purchased or not, with the universe ticker as the join key for Parts 2 and 3 and the Part 1 share in `7_Part1Share`.